<a href="https://colab.research.google.com/github/chibisova/study-tracker/blob/master/ai504_03_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3: PyTorch, Logistic Regression and MLP

- We will cover basic concepts of PyTorch Framework (tensor operations, GPU utilizing and autograd)
- We will implement simple logistic regression and multinomial logistic regression (softmax) with PyTorch
- We will use simple linear model and multi-layer perceptron (MLP) in this class

If you have any questions, feel free to ask
- For additional questions, post questions in classum.

## Why PyTorch?

- Intuitive and concise code
- Define by Run method (Tensorflow is Define and Run method)
- High compatibility with Numpy (almost one-to-one mapping)

![picture](https://drive.google.com/uc?id=1nAfTkF8Kp4YEI1pBeShs3L7NCPHx_iHQ)

## 0. Prelim: Load packages & GPU setup

In [1]:
# visualize current GPU usages in your server
!nvidia-smi

Thu Jun  4 06:09:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# set gpu by number
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # setting gpu number

In [3]:
# load packages
!pip install torch
!pip install numpy
import torch
import numpy as np

In [4]:
# print the version of PyTorch
print(torch.__version__)

2.11.0+cu128


## 1. PyTorch Tensors and Numpy

PyTorch use **tensor**: the basic data structure in PyTorch.\
**Tensor: n-dimensional array + GPU calculation is supported**\
**Almost the same with Numpy array**

![picture](https://drive.google.com/uc?id=1z2v05mGyhP_FpEa3Z4JsNpgbtEnkg0bo)

### PyTorch Tensors and Numpy shares almost identical grammer


**We will show some examples of:**
- Same operation with identical grammer
- Same operation with different grammer
- Different operation with same grammer

**We will not handle all examples in this class :(**
- For more examples, see the following reference: https://github.com/wkentaro/pytorch-for-numpy-users

**First! Define Numpy array and PyTorch tensor**

In [5]:
np_array_1 = np.array([1, 2, 3, 4])
np_array_2 = np.array([5, 6, 7, 8])
torch_tensor_1 = torch.tensor([1, 2, 3, 4])
torch_tensor_2 = torch.tensor([5 ,6 ,7, 8])

print (np_array_1)
print (np_array_2)
print (torch_tensor_1)
print (torch_tensor_2)

[1 2 3 4]
[5 6 7 8]
tensor([1, 2, 3, 4])
tensor([5, 6, 7, 8])


**1) Same operations with identical grammer**

Example) Get the shape of the tensor

In [6]:
# numpy
print (np_array_1.shape)

# torch
print (torch_tensor_1.shape)
print (torch_tensor_1.size()) # size() and shape operation is identical in torch

(4,)
torch.Size([4])
torch.Size([4])


**2) Same operations with different grammer**

Example 1) Concatenate two tensors
- numpy use `np.concatenate`
- torch use `torch.cat`
- IMPORTANT: axis (numpy) and dim (torch) is identical

In [7]:
# numpy
np_concate = np.concatenate([np_array_1, np_array_2], axis=0)
print ('----numpy----')
print (np_concate)

# torch
torch_concate= torch.cat([torch_tensor_1, torch_tensor_2], dim=0)
print ('----torch----')
print (torch_concate)

----numpy----
[1 2 3 4 5 6 7 8]
----torch----
tensor([1, 2, 3, 4, 5, 6, 7, 8])


Example 2) reshape the tensor shape
- numpy use `X.reshape`
- torch use `X.view`
- IMPORTANT: axis (numpy) and dim (torch) is identical

In [8]:
# numpy
np_reshaped = np_concate.reshape(4, 2)
print ('----numpy----')
print (np_reshaped)
print (np_reshaped.shape)

# torch
torch_reshaped = torch_concate.view(4, 2)
print ('----torch----')
print (torch_reshaped)
print (torch_reshaped.shape)

----numpy----
[[1 2]
 [3 4]
 [5 6]
 [7 8]]
(4, 2)
----torch----
tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])
torch.Size([4, 2])


**3) Different operations with same grammer (Confusing operations)**

Example) manipulation tensors
- Same grammer `repeat`  has different operations

In [9]:
x = np.array([1, 2, 3])
x_repeat = x.repeat(2)

print ('----numpy----')
print (x)
print (x_repeat)

x = torch.tensor([1, 2, 3])
x_repeat = x.repeat(2)

print ('----torch----')
print (x)
print (x_repeat)

# To obtain the same result with np.repeat (will skip explanation: you should be proficient with reshaping operations)
print('----obtain the same result-----')
x_repeat = x.view(3, 1)
print (x_repeat)

x_repeat = x_repeat.repeat(1, 2)
print (x_repeat)

x_repeat = x_repeat.view(-1)
print (x_repeat)

----numpy----
[1 2 3]
[1 1 2 2 3 3]
----torch----
tensor([1, 2, 3])
tensor([1, 2, 3, 1, 2, 3])
----obtain the same result-----
tensor([[1],
        [2],
        [3]])
tensor([[1, 1],
        [2, 2],
        [3, 3]])
tensor([1, 1, 2, 2, 3, 3])


In [ ]:
# similar manipulation operation: stack & repeat
x = torch.tensor([1, 2, 3])
x_repeat = x.repeat(4)
x_stack = torch.stack([x, x, x, x])

print (x_repeat)
print (x_stack)
print (x_repeat.view(4, 3)) # reshape x

tensor([1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3])
tensor([[1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3]])
tensor([[1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3]])


## 2. Tensor operations under GPU utilization

Deep learning frameworks utilize GPUs to accelarate computations.

In this section, we will learn **how to utilize GPU** in PyTorch

In [10]:
print(torch.cuda.is_available())  # Is GPU accessible?

True


In [11]:
a = torch.ones(3)
b = torch.randn(100, 50, 3)

In [12]:
print(a.device)
print(b.device)

cpu
cpu


In [13]:
c = a + b

In [14]:
print(c.device)

cpu


In [15]:
# upload a and b to GPU
a = a.to('cuda')
b = b.to('cuda')

In [16]:
print(a.device)
print(b.device)

cuda:0
cuda:0


In [17]:
c = a + b

In [18]:
print(c.device)

cuda:0


In [19]:
c = c.to('cpu')

In [20]:
print(c.device)

cpu


## 3. Autograd

Central to all neural networks in PyTorch is the `autograd` package.

The `autograd` package provides automatic differentiation for all operations on Tensors.

`torch.Tensor` is the central class of the package. If you set its attribute `.requires_grad` as True, it starts to track all operations on it. When you finish your computation you can call `.backward()` and have all the gradients computed automatically. The gradient for this tensor will be accumulated into `.grad` attribute.

To stop a tensor from tracking history, you can call `.detach()` to detach it from the computation history, and to prevent future computation from being tracked.

### Example

In [21]:
x = torch.ones(2, 2, requires_grad=True)
print(x)

tensor([[1., 1.],
        [1., 1.]], requires_grad=True)


In [22]:
y = x + 2
print(y)

tensor([[3., 3.],
        [3., 3.]], grad_fn=<AddBackward0>)


In [23]:
z = y * y * 3
print(z)

tensor([[27., 27.],
        [27., 27.]], grad_fn=<MulBackward0>)


In [24]:
out = z.mean()
print(out)

tensor(27., grad_fn=<MeanBackward0>)


In [26]:
y.retain_grad()
z.retain_grad()
out.backward()

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

![picture](https://drive.google.com/uc?id=1JyMWTbaU6ktJAHx2XqiU7s4tId-cxiLF)
![picture](https://drive.google.com/uc?id=17j-aNqj1yjZfVPCKZJRt6YVZ-7usf5PH)

In [27]:
print(z.grad)

tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])


![picture](https://drive.google.com/uc?id=1jPfdq6piSkkwZ21nX7kIBa-xGJE6uPBu)
![picture](https://drive.google.com/uc?id=1NN0kpdvRRP9NwguXJHnU3u8VikMFUKw2)

In [28]:
print(y.grad)

tensor([[4.5000, 4.5000],
        [4.5000, 4.5000]])


![picture](https://drive.google.com/uc?id=1HllHu2CxuNFX8mc6QdQEEtnXJ3Rvo6TE)
![picture](https://drive.google.com/uc?id=1jWJPOXVLG6mdUyDSklocNWPVa9Rg62K3)

In [29]:
print(x.grad)

tensor([[4.5000, 4.5000],
        [4.5000, 4.5000]])


### Efficient inference (testing) with torch.no_grad()

To prevent tracking history (and using memory), you can also wrap the code block in with `torch.no_grad()`

Situation: when **gradient calculation is not required** e.g., inference\
Solution: use `torch.no_grad()`, then torch doesn't generate computational graph for back propagation, therefore it is **much faster**

In [30]:
with torch.no_grad():
    x = torch.ones(2, 2, requires_grad=True)
    y = x + 2
    z = y * y * 3
    out = z.mean()

In [31]:
out

tensor(27.)

In [32]:
out.backward() ## ERROR!!!!: we used torch.no_grad()!!

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

## 4. nn.Module

![picture](https://drive.google.com/uc?id=1Vu3oRATA-EWDycO2zVWkBdzndU-8C5cB)

### Using pre-defined modules (subset of models) in PyTorch

In [33]:
import torch.nn as nn

X = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

print (X)
print (X.shape)

tensor([[1., 2., 3.],
        [4., 5., 6.]])
torch.Size([2, 3])


In [34]:
# input dim 3, output dim 1
linear_fn = nn.Linear(3, 1)

In [35]:
linear_fn  # WX + b

Linear(in_features=3, out_features=1, bias=True)

In [36]:
Y = linear_fn(X)
print(Y)
print(Y.shape)

tensor([[1.4073],
        [2.6880]], grad_fn=<AddmmBackward0>)
torch.Size([2, 1])


In [37]:
Y = Y.sum()
print(Y)

tensor(4.0953, grad_fn=<SumBackward0>)


You can use other types of `nn.Module` in PyTorch

In [ ]:
nn.Conv2d
nn.RNNCell
nn.LSTMCell
nn.GRUCell
nn.Transformer;

### How can we design a customized model (neural network)?

In [38]:
class Model(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim):
        super(Model, self).__init__()
        self.linear_1 = nn.Linear(input_dim, hidden_dim)
        self.linear_2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.linear_1(x)
        x = self.relu(x) # Activation function
        x = self.linear_2(x)
        return x

**What is activation function?**
- They make non-linearity for deep neural networks
- Therefore, deep neural networks can approximate complex functions

In [ ]:
nn.Sigmoid
nn.ReLU
nn.LeakyReLU
nn.Tanh;

## 5. MNIST classification with PyTorch (Logistic regression & MLP)

### What is MNIST & How to do multi-class classification?

The MNIST database of **handwritten digits from 0 to 9**, has a training set of 60,000 examples, and a test set of 10,000 examples.

Since we have 10 classes (0~9), current problem can be interpreted as **multinomial logistic regression** (**multi-class classification**).

Therefore, we use **softmax** function to handle multiple class output with **cross-entropy** loss function.

![picture](https://drive.google.com/uc?id=1v-QvM2MEMku6wWMb_8f8NIqIDzby7wJP)

### 1. Binary Classification

**What it is:** This is the simplest form of classification where there are only two possible output classes. The model predicts one of two outcomes.

**How to identify:** If your target variable has exactly two distinct categories or states.

**Examples:**
*   **Spam detection:** An email is either 'spam' or 'not spam'.
*   **Disease diagnosis:** A patient either 'has the disease' or 'does not have the disease'.
*   **Customer churn:** A customer will either 'churn' or 'not churn'.

**Typical Machine Learning Approach:**
*   **Output Layer Activation:** Sigmoid function (outputs a probability between 0 and 1).
*   **Loss Function:** Binary Cross-Entropy (BCE) Loss.

### 2. Multi-Class Classification

**What it is:** This is when there are more than two possible output classes, and each instance belongs to *exactly one* of these classes.

**How to identify:** If your target variable has more than two distinct categories, and an instance can only be assigned to one of them.

**Examples:**
*   **MNIST Digit Recognition (as in your notebook):** An image of a digit is either a '0', '1', '2', ..., '9' – it cannot be both a '1' and a '7' simultaneously.
*   **Animal classification:** An image contains a 'cat', 'dog', or 'bird' – it's one or the other.
*   **Sentiment analysis (fine-grained):** A review is 'very negative', 'negative', 'neutral', 'positive', or 'very positive'.

**Typical Machine Learning Approach:**
*   **Output Layer Activation:** Softmax function (outputs a probability distribution over all classes, summing to 1).
*   **Loss Function:** Categorical Cross-Entropy Loss (also known as Softmax Cross-Entropy Loss or `nn.CrossEntropyLoss` in PyTorch, which combines softmax and NLL loss).

### 3. Multi-Label Classification

**What it is:** This is when there are multiple possible output classes, and an instance can belong to *zero, one, or multiple* classes simultaneously.

**How to identify:** If your target variable consists of multiple independent binary choices. An instance can have multiple 'labels' associated with it.

**Examples:**
*   **Image tagging:** An image might contain a 'dog', 'grass', and 'sky' – all at the same time.
*   **Movie genres:** A movie can be 'action', 'comedy', and 'sci-fi' simultaneously.
*   **Document classification:** A document might be about 'politics', 'economy', and 'social issues'.

**Typical Machine Learning Approach:**
*   **Output Layer Activation:** Sigmoid function for *each* output class (each output represents the probability of that specific label being present, independently).
*   **Loss Function:** Binary Cross-Entropy Loss applied independently for each label (often called `nn.BCEWithLogitsLoss` in PyTorch for numerical stability).

### Summary for Identifying Problem Type:

To determine the type of classification problem, ask yourself:

*   **How many categories are there?** (If 2, likely binary; if > 2, multi-class or multi-label).
*   **Can an instance belong to more than one category simultaneously?**
    *   **No:** It's a **multi-class** problem (like MNIST digits).
    *   **Yes:** It's a **multi-label** problem (like image tags).

## How do we choose a loss function?

Choosing the right loss function is crucial for training effective machine learning models, as it quantifies the error between predicted and actual values, guiding the model's learning process. The choice primarily depends on the type of problem you're trying to solve and the nature of your model's output.

### General Principle

The loss function mathematically penalizes the model for making incorrect predictions. The goal of training is to minimize this loss function, thereby making the model's predictions as accurate as possible.

### Key Considerations:

1.  **Problem Type:** Is it a regression, binary classification, multi-class classification, or multi-label classification problem?
2.  **Output Layer Activation:** What kind of output does your model's final layer produce (e.g., raw logits, probabilities, continuous values)?
3.  **Data Distribution/Characteristics:** Are there outliers? Is the data imbalanced?

### Common Loss Functions by Problem Type:

#### 1. Regression Problems

**Goal:** Predict a continuous numerical value.

*   **Mean Squared Error (MSE) / L2 Loss:**
    *   **Formula:** Average of the squared differences between predicted and actual values. $MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$
    *   **When to use:** Most common choice for regression. It heavily penalizes large errors, making it sensitive to outliers.
    *   **PyTorch:** `nn.MSELoss()`

*   **Mean Absolute Error (MAE) / L1 Loss:**
    *   **Formula:** Average of the absolute differences between predicted and actual values. $MAE = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|$
    *   **When to use:** Less sensitive to outliers than MSE, as it doesn't square the errors. Useful when outliers are present and you don't want them to disproportionately affect the training.
    *   **PyTorch:** `nn.L1Loss()`

*   **Huber Loss (Smooth L1 Loss):**
    *   **When to use:** A hybrid of MSE and MAE. It's quadratic for small errors (like MSE) and linear for large errors (like MAE). This makes it less sensitive to outliers than MSE while still providing a smooth gradient for optimization. Good for general regression tasks.
    *   **PyTorch:** `nn.SmoothL1Loss()`

#### 2. Binary Classification Problems

**Goal:** Predict one of two classes (e.g., 0 or 1, true or false).

*   **Binary Cross-Entropy (BCE) Loss:**
    *   **Formula:** Measures the performance of a classification model whose output is a probability value between 0 and 1. For a single sample: $-[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$
    *   **When to use:** Standard for binary classification problems where the model's output layer uses a **Sigmoid activation function** to produce probabilities.
    *   **PyTorch:** `nn.BCELoss()` (expects sigmoid output) or `nn.BCEWithLogitsLoss()` (combines sigmoid and BCE for numerical stability, expects raw logits output).

#### 3. Multi-Class Classification Problems

**Goal:** Predict one class out of *more than two* mutually exclusive classes (e.g., MNIST digits 0-9, where an image is *only* one digit).

*   **Categorical Cross-Entropy Loss:**
    *   **Formula:** For a single sample, it's typically $-\sum_{c=1}^{C} y_{o,c} \log(\hat{y}_{o,c})$, where $y_{o,c}$ is a binary indicator (0 or 1) if class $c$ is the correct classification for observation $o$, and $\hat{y}_{o,c}$ is the predicted probability of observation $o$ being of class $c$.
    *   **When to use:** The default choice for multi-class classification where the model's output layer uses a **Softmax activation function** to produce a probability distribution over all classes.
    *   **PyTorch:** `nn.CrossEntropyLoss()` (This is a very convenient loss in PyTorch as it internally applies `LogSoftmax` and `NLLLoss`, so you should pass raw logits from your model's final layer, *not* probabilities from a softmax activation).

#### 4. Multi-Label Classification Problems

**Goal:** Predict zero, one, or multiple classes out of *more than two* possible classes (e.g., an image can contain both a 'dog' and 'grass').

*   **Binary Cross-Entropy (BCE) Loss (applied per label):**
    *   **When to use:** Each label is treated as an independent binary classification problem. Your model's output layer should have a **Sigmoid activation function for each output neuron** (one for each possible label). The loss is then computed for each label independently and summed or averaged.
    *   **PyTorch:** `nn.BCEWithLogitsLoss()` (This is typically preferred as it's more numerically stable than `nn.BCELoss` followed by a `Sigmoid`).

### How to Decide for Your Problem:

1.  **Look at your target variable:**
    *   Is it continuous? -> **Regression** (MSE, MAE, Huber).
    *   Is it categorical?
        *   Only two categories? -> **Binary Classification** (BCE).
        *   More than two categories, but only one can be true at a time? -> **Multi-Class Classification** (Categorical Cross-Entropy).
        *   More than two categories, and multiple can be true at once? -> **Multi-Label Classification** (BCE per label).

2.  **Consider your model's final activation:** Make sure your chosen loss function is compatible with the activation function of your model's output layer. For example, `nn.CrossEntropyLoss` in PyTorch expects raw logits, not probabilities, because it applies softmax internally.

### Load packages

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

### Load datasets for training & testing

In [3]:
# MNIST dataset
train_dataset = torchvision.datasets.MNIST(root='./', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = torchvision.datasets.MNIST(root='./', train=False, transform=transforms.ToTensor())

# Data loader
# mini batch size
train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=128, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 61.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.69MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.4MB/s]


### Define model (we will use one layer classifier first)

In [42]:
# Define model class
# This model has one hidden layer
class Multinomial_logistic_regression(nn.Module):
    def __init__(self, input_size, output_size):
        super(Multinomial_logistic_regression, self).__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, x):
        out = self.fc(x)
        return out

In [43]:
# Generate model
model = Multinomial_logistic_regression(784, 10)  # init(784, 10)
# input dim: 784  / output dim: 10

In [44]:
model

Multinomial_logistic_regression(
  (fc): Linear(in_features=784, out_features=10, bias=True)
)

In [45]:
# Upload model to GPU
model = model.to('cuda')

### Define optimizer

Optimization is about finding the best solution (model parameter) that fits the given dataset!

PyTorch optimizer is about **which optimization methods to use for training**

We will not handle the details in this class. (take **"Optimization for AI (AI505)"** course)

In [47]:
# Optimizer define
# optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
# toptimizer = orch.optim.Adam(model.parameters(), lr=0.05)

![picture](https://drive.google.com/uc?id=1BvkB6O1hsGZ4YkD92k-E3I59omprN7qz)

### Train the model

In [51]:
# Loss function define (we use cross-entropy)
loss_fn = nn.CrossEntropyLoss()

#Train the model
total_step = len(train_loader)

for epoch in range(10):
    for i, (images, labels) in enumerate(train_loader):  # mini batch for loop
        # upload to gpu
        images = images.reshape(-1, 28*28).to('cuda')
        #print(f"Shape after reshape: {images.shape}") # Added this line to check shape
        labels = labels.to('cuda')

        # Forward
        outputs = model(images)  # forwardI(images): get prediction
        loss = loss_fn(outputs, labels)  # calculate the loss (cross entropy loss) with ground truth & prediction value

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()  # automatic gradient calculation (autograd)
        optimizer.step()  # update model parameter with requires_grad=True

        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                   .format(epoch+1, 10, i+1, total_step, loss.item()))

Epoch [1/10], Step [100/469], Loss: 0.2320
Epoch [1/10], Step [200/469], Loss: 0.4791
Epoch [1/10], Step [300/469], Loss: 0.1642
Epoch [1/10], Step [400/469], Loss: 0.2234
Epoch [2/10], Step [100/469], Loss: 0.1671
Epoch [2/10], Step [200/469], Loss: 0.2018
Epoch [2/10], Step [300/469], Loss: 0.1781
Epoch [2/10], Step [400/469], Loss: 0.3346
Epoch [3/10], Step [100/469], Loss: 0.1879
Epoch [3/10], Step [200/469], Loss: 0.1899
Epoch [3/10], Step [300/469], Loss: 0.3445
Epoch [3/10], Step [400/469], Loss: 0.3568
Epoch [4/10], Step [100/469], Loss: 0.2203
Epoch [4/10], Step [200/469], Loss: 0.4851
Epoch [4/10], Step [300/469], Loss: 0.2395
Epoch [4/10], Step [400/469], Loss: 0.4213
Epoch [5/10], Step [100/469], Loss: 0.1850
Epoch [5/10], Step [200/469], Loss: 0.4746
Epoch [5/10], Step [300/469], Loss: 0.2597
Epoch [5/10], Step [400/469], Loss: 0.2683
Epoch [6/10], Step [100/469], Loss: 0.2142
Epoch [6/10], Step [200/469], Loss: 0.2545
Epoch [6/10], Step [300/469], Loss: 0.1834
Epoch [6/10

### Test the model

In [52]:
# Test the model
# In test phase, we don't need to compute gradients (for memory efficiency)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.reshape(-1, 28*28).to('cuda')
        labels = labels.to('cuda')
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)  # classification -> get the label prediction of top 1
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the 10000 test images: {} %'.format(100 * correct / total))

Accuracy of the network on the 10000 test images: 92.32 %


### New model: MLP (multi-layer-perceptron)

Previous model used multinomial logistic regression (one linear layer)\
What if we use **MLP (multi-layer-perceptron)?** A neural network with hidden layers?

In [6]:
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size) # Renamed final layer to maintain numbering
        self.sigmoid = nn.Sigmoid()  # sigmoid activation function (you can customize)
        self.relu = nn.LeakyReLU()  # relu activation function (you can customize)
        self.tanh = nn.Tanh()  # tanh activation function (you can customize)
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out) # Pass through the new layer
        return out

In [10]:
# Generate model
model = NeuralNet(784, 200, 10)  # init(784, 20, 10)
# input dim: 784  / hidden dim: 20  / output dim: 10

# Upload model to GPU
model = model.to('cuda')

# Loss function define (we use cross-entropy)
loss_fn = nn.CrossEntropyLoss()

# Define optimizer
# optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
# optimizer = torch.optim.ASGD(model.parameters(), lr=0.05)

# Train the model
total_step = len(train_loader)

num_epochs = 30

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):  # mini batch for loop
        # upload to gpu
        images = images.reshape(-1, 28*28).to('cuda')
        labels = labels.to('cuda')

        # Forward
        outputs = model(images)  # forwardI(images): get prediction
        loss = loss_fn(outputs, labels)  # calculate the loss (cross entropy loss) with ground truth & prediction value

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()  # automatic gradient calculation (autograd)
        optimizer.step()  # update model parameter with requires_grad=True

        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

Epoch [1/30], Step [100/469], Loss: 0.3179
Epoch [1/30], Step [200/469], Loss: 0.2570
Epoch [1/30], Step [300/469], Loss: 0.1856
Epoch [1/30], Step [400/469], Loss: 0.1959
Epoch [2/30], Step [100/469], Loss: 0.1658
Epoch [2/30], Step [200/469], Loss: 0.1143
Epoch [2/30], Step [300/469], Loss: 0.1923
Epoch [2/30], Step [400/469], Loss: 0.0767
Epoch [3/30], Step [100/469], Loss: 0.0327
Epoch [3/30], Step [200/469], Loss: 0.0564
Epoch [3/30], Step [300/469], Loss: 0.0369
Epoch [3/30], Step [400/469], Loss: 0.0370
Epoch [4/30], Step [100/469], Loss: 0.0659
Epoch [4/30], Step [200/469], Loss: 0.0412
Epoch [4/30], Step [300/469], Loss: 0.0925
Epoch [4/30], Step [400/469], Loss: 0.0153
Epoch [5/30], Step [100/469], Loss: 0.0198
Epoch [5/30], Step [200/469], Loss: 0.0643
Epoch [5/30], Step [300/469], Loss: 0.0508
Epoch [5/30], Step [400/469], Loss: 0.0256
Epoch [6/30], Step [100/469], Loss: 0.0456
Epoch [6/30], Step [200/469], Loss: 0.0214
Epoch [6/30], Step [300/469], Loss: 0.0350
Epoch [6/30

### Learning Rate Scheduler

Learning rate schedulers adjust the learning rate during training, which can help in achieving better convergence and preventing the model from getting stuck in local minima or overshooting.

Here, we're using `torch.optim.lr_scheduler.StepLR`:
- `step_size`: The number of epochs after which the learning rate will be decayed.
- `gamma`: The factor by which the learning rate will be reduced (e.g., `0.1` means the learning rate becomes 10% of its previous value).

In each epoch, after `optimizer.step()` is called, we also call `scheduler.step()` to update the learning rate according to the defined schedule.

In [ ]:
# Define a learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print("Training with learning rate scheduler...")

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = images.reshape(-1, 28*28).to('cuda')
        labels = labels.to('cuda')

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Current LR: {:.6f}'
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item(), scheduler.get_last_lr()[0]))

    # Update the learning rate scheduler after each epoch
    scheduler.step()

In [11]:
# Test the model
# In test phase, we don't need to compute gradients (for memory efficiency)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.reshape(-1, 28*28).to('cuda')
        labels = labels.to('cuda')
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)  # classification -> get the label prediction of top 1
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the 10000 test images: {} %'.format(100 * correct / total))

Accuracy of the network on the 10000 test images: 98.35 %


### Change the following options to obtain better accuracy!! (try it by your-self)

#### (1) Model configurations:
- size of hidden layer units
- number of layers
- type of activation function (e.g., relu, tanh, softplus etc.)

#### (2) Optimization configurations
- learning rate
- epoch
- type of optimizer
- momentem hyperparameter

# Testing:

Original setup : 94.74%
20 neurons, 3 layers, sigmoid, lr = 0.05, 10 epochs, SGD, momentum = 0.9

*   20 vs 40 neurons in hidden layer: 94.74% -> 96.26%
*   3 -> 4 layers: 93.74%
*   sigmoid -> Tanh activation: 95.69%
*   learning rate 0.05 -> 0.01: 90.08%
*   10 -> 20 -> 40 epochs: 95.93% -> 95.69% No much effect
*   Adam -> ASGD optimizer: 92.86% -> 83.16%
*   Momentem hyperparameter 0.99 -> 0.8 -> 0.1 : 94.85% -> 94.18% -> 84.81%

Exp1:


*   Neurons: 200
*   Momentum: 0.99
*   Epochs: 20
-> 97.4%

Exp2:
*   Neurons: 200
*   Momentum: 0.99
*   Epochs: 20
*   Relu -> 9%

Caused by momentum 0.99 -> lower it to 0.9

Updated result:
98.26%

Exp3:
*   Neurons: 200
*   Momentum: 0.9
*   Epochs: 20
*   Tanh

Results: 98.36%

Exp3:
*   Neurons: 200
*   Momentum: 0.9
*   Epochs: 20
*   LeakyReLU

Results: 98.4%

Exp3:
*   Neurons: 200
*   Momentum: 0.9
*   Epochs: 20
*   LeakyReLU
*   Learning rate 0.1
-> 98.13%

Exp4:

*   Neurons: 200
*   Momentum: 0.9
*   Epochs: 30
*   LeakyReLU
*   Learning rate 0.1
-> 98.35%




In [11]:
import torch.optim as optim

print("Available PyTorch Optimizers:")
for name in dir(optim):
    if name.startswith(('Ad', 'ASGD', 'Adam', 'LBFGS', 'NAdam', 'Optimizer', 'Rprop', 'RMSprop', 'SGD')):
        obj = getattr(optim, name)
        if isinstance(obj, type) and issubclass(obj, optim.Optimizer):
            print(f"- {name}")

Available PyTorch Optimizers:
- ASGD
- Adadelta
- Adafactor
- Adagrad
- Adam
- AdamW
- Adamax
- LBFGS
- NAdam
- Optimizer
- RMSprop
- Rprop
- SGD
